# From WiFeS data cube to stellar parameters

## Winter-school practical: stellar spectroscopy with WiFeS

In this practical you will turn a reduced WiFeS data cube into a one-dimensional stellar spectrum, compare it with synthetic spectra, estimate a radial velocity, and obtain a **rough** set of stellar parameters.

The aim is not to build a production-quality pipeline in one afternoon. The aim is to understand:

1. what information a stellar spectrum contains;
2. which analysis choices are being made;
3. how those choices affect the answer; and
4. how to make plots that communicate the result honestly.

Work in pairs. Discuss each **checkpoint** before moving on. Optional challenges are included for groups that finish early.

## Suggested timing

| Section | Approximate time |
|---|---:|
| 0. What does a spectrum mean? | 15 min |
| 1. Explore synthetic spectra | 30 min |
| 2. Inspect and extract a WiFeS spectrum | 40 min |
| 3. Join the blue and red arms | 25 min |
| 4. Estimate the radial velocity | 30 min |
| 5. Compare with synthetic spectra | 45 min |
| Breaks | 20 min total |
| 6. Plot clinic and interpretation | 30 min |
| 7. Share results | 15 min |

# 0. Before coding: what does a stellar spectrum tell us?

A spectrum is a measurement of flux as a function of wavelength. Its broad shape and individual features can carry information about:

- stellar temperature;
- surface gravity and evolutionary state;
- chemical composition;
- motion along the line of sight;
- rotation and other forms of line broadening;
- dust extinction between the star and us;
- the atmosphere, instrument, calibration, and noise.

### Prediction exercise

Before looking at the models, discuss and write down what you expect to happen when:

1. temperature increases;
2. surface gravity decreases;
3. iron abundance increases;
4. carbon abundance increases;
5. extinction increases;
6. the star moves away from us.

There is no penalty for an incorrect prediction. The point is to make your assumptions explicit before seeing the answer.

**Your predictions**

- Increasing temperature:
- Decreasing surface gravity:
- Increasing [Fe/H]:
- Increasing [C/Fe]:
- Increasing $E(B-V)$:
- Positive radial velocity:

# Setup

The practical expects the same directory structure as the supplied course data:

```text
data/
    grid_3000-8000_res1.0-absfluxes.fits
    Reddening_Rv_3_1.txt
    <your blue WiFeS cube>
    <your red WiFeS cube>
figures/
```

Adjust the paths in the configuration cell when necessary.

In [ ]:
from pathlib import Path

import numpy as np

import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table
from astropy.constants import c
import astropy.units as u

try:
    %matplotlib inline
    %config InlineBackend.figure_format='retina'
except:
    pass

plt.rcParams.update({
    "figure.figsize": (10, 4),
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
})

ROOT = Path(".")
SYNTHETIC_GRID_FILE = ROOT / "data/grid_3000-8000_res1.0-absfluxes.fits"
REDDENING_FILE = ROOT / "data/Reddening_Rv_3_1.txt"

# Replace these with the files / or uncomment the later ones to read in different data.
BLUE_FITS_FILE = ROOT / "data/T2m3wb-20220914.090809-0218.fits"
RED_FITS_FILE  = ROOT / "data/T2m3wr-20220914.090809-0218.fits"

# BLUE_FITS_FILE = ROOT / "data/OBK-1173216-WiFeS-Blue-UT20250904T094414-4.cube.fits"
# RED_FITS_FILE  = ROOT / "data/OBK-1173216-WiFeS-Red--UT20250904T094414-4.cube.fits"

# BLUE_FITS_FILE = ROOT / "data/OBK-1175488-WiFeS-Blue-UT20250904T091726-3.cube.fits"
# RED_FITS_FILE  = ROOT / "data/OBK-1175488-WiFeS-Red--UT20250904T091726-3.cube.fits"

FIGURE_DIR = ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Check the course-data directory and the path above."
        )
    return path

# 1. Explore synthetic stellar spectra

Synthetic spectra let us change one physical quantity at a time. This is useful for learning, although real stars do not always vary one parameter independently of all others.

The supplied grid contains spectra at discrete values of:

- effective temperature $T_{\mathrm{eff}}$;
- surface gravity $\log g$;
- iron abundance [Fe/H];
- carbon-to-iron abundance [C/Fe];
- a broadening parameter;
- extinction $E(B-V)$, applied after loading the grid.

In [ ]:
def read_synthetic_grid(grid_file):
    """Read the supplied grid of absolute-flux synthetic spectra."""
    grid_file = require_file(grid_file)
    metadata = fits.getdata(grid_file, ext=1)[0]
    flux = fits.getdata(grid_file, ext=2)

    return {
        "WAVE": metadata["WAVE"],
        "TEFF": metadata["TEFF"],
        "LOGG": metadata["LOGG"],
        "FEH": metadata["FEH"],
        "CFE": metadata["ABUND"],
        "BROAD": metadata["BROAD"],
        "FLUX": flux,
    }

require_file(REDDENING_FILE)
wave_reddening, extinction_curve = np.loadtxt(
    REDDENING_FILE, usecols=(0, 1), unpack=True
)

def apply_reddening(wavelength, flux, ebv):
    attenuation = np.interp(wavelength, wave_reddening, extinction_curve)
    transmission = 10 ** (-0.4 * attenuation * ebv)
    return flux * transmission

def get_synthetic_spectrum(
    grid, teff, logg, fe_h, c_fe=0.0, broad=100.0, ebv=0.0
):
    """Return one exact grid spectrum. This function does not interpolate."""
    match = np.where(
        (grid["TEFF"] == teff)
        & (grid["LOGG"] == logg)
        & (grid["FEH"] == fe_h)
        & (grid["CFE"] == c_fe)
        & (grid["BROAD"] == broad)
    )[0]

    if len(match) != 1:
        raise ValueError(
            "No unique spectrum exists for this exact parameter combination. "
            "Inspect the available grid values below."
        )

    flux = grid["FLUX"][match[0]]
    return apply_reddening(grid["WAVE"], flux, ebv)

grid = read_synthetic_grid(SYNTHETIC_GRID_FILE)
print(f"Grid spectra: {len(grid['TEFF'])}")
print(f"Wavelength pixels: {len(grid['WAVE'])}")

## 1.1 Inspect the available grid

Do not guess parameter values that are absent from the grid. Print the unique values and identify:

- the temperature range;
- the surface-gravity range;
- the metallicity range;
- the available carbon abundances;
- the available broadening values.

In [ ]:
for name in ["TEFF", "LOGG", "FEH", "CFE", "BROAD"]:
    values = np.unique(grid[name])
    print(f"{name:>5}: {values}")

## 1.2 Change one parameter at a time

The helper below plots several models while keeping the other parameters fixed.

Choose valid values from the grid and investigate:

1. temperature;
2. surface gravity;
3. [Fe/H];
4. [C/Fe];
5. $E(B-V)$.

For each comparison, inspect both the **overall spectral shape** and a **short wavelength interval** containing individual lines.

In [ ]:
def compare_synthetic_spectra(
    grid,
    varying_parameter,
    values,
    fixed_parameters,
    wavelength_limits=(3500, 7000),
):
    """Plot spectra while changing one selected parameter."""
    allowed = {"teff", "logg", "fe_h", "c_fe", "broad", "ebv"}
    if varying_parameter not in allowed:
        raise ValueError(f"varying_parameter must be one of {sorted(allowed)}")

    fig, ax = plt.subplots()

    for value in values:
        parameters = dict(fixed_parameters)
        parameters[varying_parameter] = value
        flux = get_synthetic_spectrum(grid, **parameters)
        ax.plot(grid["WAVE"], flux, label=f"{varying_parameter} = {value}")

    ax.set_xlim(*wavelength_limits)
    ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
    ax.set_ylabel("Flux [absolute model scale]")
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
# Example: temperature. Change the values or wavelength interval after the first run.
fixed = dict(teff=6000, logg=4.5, fe_h=-1.0, c_fe=0.0, broad=100.0, ebv=0.0)

compare_synthetic_spectra(
    grid,
    varying_parameter="teff",
    values=[5000, 6000, 7000],  # TODO: choose valid values
    fixed_parameters=fixed,
    wavelength_limits=(3500, 7000),
)

In [ ]:
# TODO: repeat the comparison for logg, fe_h, c_fe, and ebv.
# You may copy the call above. Change only one quantity at a time.

# compare_synthetic_spectra(...)

### Checkpoint 1: interpret the models

Discuss:

- Which parameter most strongly changes the broad spectral shape?
- Which parameters mainly change absorption features?
- Where are temperature and extinction partly degenerate?
- Why might surface gravity be easier to measure from some lines than from the full spectrum?
- Did any result contradict your prediction?

**Checkpoint 1 response**

...

# 2. Inspect and extract the WiFeS observation

A WiFeS data cube has two spatial dimensions and one wavelength dimension. For a point source, the star occupies several spatial pixels because of atmospheric seeing and the instrument response.

Your tasks are to:

1. inspect a white-light image;
2. choose a spatial extraction aperture;
3. sum the stellar signal;
4. estimate a rough signal-to-noise ratio.

This is a deliberately simple extraction. A production pipeline would treat sky subtraction, variance propagation, bad pixels, wavelength-dependent centring, and optimal extraction more carefully.

In [ ]:
file = fits.open(BLUE_FITS_FILE)
file.info()

In [ ]:
def read_wifes_cube(filename):
    filename = require_file(filename)
    with fits.open(filename) as hdul:
        data = np.asarray(hdul[0].data, dtype=float)
        header = hdul[0].header.copy()

    n_wave = header["NAXIS3"]
    wavelength = header["CRVAL3"] + header["CDELT3"] * np.arange(n_wave)
    object_name = header.get("OBJNAME", filename.stem)

    if data.shape[0] != n_wave:
        raise ValueError(
            f"Expected wavelength to be the first NumPy axis, but data shape is {data.shape} "
            f"and NAXIS3={n_wave}. Ask an instructor to inspect the FITS axis order."
        )

    return {
        "filename": filename,
        "object": object_name,
        "wavelength": wavelength,
        "cube": data,
    }

blue = read_wifes_cube(BLUE_FITS_FILE)
red = read_wifes_cube(RED_FITS_FILE)

print("Object:", blue["object"])
print("Blue cube shape:", blue["cube"].shape)
print("Red cube shape:", red["cube"].shape)

## 2.1 Locate the star

Create a white-light image by taking the median along the wavelength axis.

Questions to consider:

- Where is the star?
- Is there a background gradient?
- Are there other sources?
- Does the source position differ between the blue and red cubes?
- Which displayed axis is $x$ and which is $y$?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

blue_white_light = np.nanmedian(blue["cube"], axis=0)
red_white_light = np.nanmedian(red["cube"], axis=0)

axes[0].imshow(blue_white_light, origin="lower", aspect="equal", cmap="Greys_r")
axes[0].set_title("Blue arm: white-light image")
axes[0].set_xlabel("Spatial pixel")
axes[0].set_ylabel("Spatial pixel")

axes[1].imshow(red_white_light, origin="lower", aspect="equal", cmap="Greys_r")
axes[1].set_title("Red arm: white-light image")
axes[1].set_xlabel("Spatial pixel")
axes[1].set_ylabel("Spatial pixel")

fig.tight_layout()
plt.show()

## 2.2 Choose extraction apertures

Set the aperture boundaries after inspecting the images. The starting values below are only placeholders.

A larger aperture captures more stellar light, but also more sky and detector noise. A smaller aperture can lose stellar light and may distort the spectral shape if the centroid changes with wavelength.

In [ ]:
# TODO: adjust these boundaries for your source.
# Python slices include the start value but exclude the end value.
blue_aperture = dict(y_start=9, y_end=15, x_start=5, x_end=10)
red_aperture  = dict(y_start=9, y_end=15, x_start=5, x_end=10)

def draw_aperture(ax, aperture):
    from matplotlib.patches import Rectangle

    rectangle = Rectangle(
        (aperture["x_start"], aperture["y_start"]),
        aperture["x_end"] - aperture["x_start"],
        aperture["y_end"] - aperture["y_start"],
        fill=False,
        linewidth=2,
        color = 'C0'
    )
    ax.add_patch(rectangle)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(blue_white_light, origin="lower", aspect="equal", cmap="Greys_r")
draw_aperture(axes[0], blue_aperture)
axes[0].set_title("Blue extraction aperture")

axes[1].imshow(red_white_light, origin="lower", aspect="equal", cmap="Greys_r")
draw_aperture(axes[1], red_aperture)
axes[1].set_title("Red extraction aperture")

fig.tight_layout()
plt.show()

In [ ]:
def extract_box_spectrum(cube, aperture):
    spatial_cutout = cube[
        :,
        aperture["y_start"]:aperture["y_end"],
        aperture["x_start"]:aperture["x_end"],
    ]
    return np.nansum(spatial_cutout, axis=(1, 2))

blue_flux = extract_box_spectrum(blue["cube"], blue_aperture)
red_flux = extract_box_spectrum(red["cube"], red_aperture)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(blue["wavelength"], blue_flux)
axes[0].set_title("Extracted blue spectrum")
axes[0].set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
axes[0].set_ylabel("Summed flux")

axes[1].plot(red["wavelength"], red_flux)
axes[1].set_title("Extracted red spectrum")
axes[1].set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
axes[1].set_ylabel("Summed flux")

fig.tight_layout()
plt.show()

## 2.3 Estimate a rough signal-to-noise ratio

We will use a line-poor wavelength interval and estimate

$$
\mathrm{S/N} \approx
\frac{\mathrm{median}(f)}
{0.5\,(P_{84}(f)-P_{16}(f))}.
$$

This does **not** distinguish photon noise from real weak absorption lines or calibration structure. It is only a practical approximation for this exercise.

Inspect the proposed windows and move them if they contain strong features or lie outside your wavelength coverage.

In [ ]:
def estimate_snr(wavelength, flux, window):
    use = (
        (wavelength >= window[0])
        & (wavelength <= window[1])
        & np.isfinite(flux)
    )
    if np.count_nonzero(use) < 10:
        raise ValueError(f"Too few valid pixels in S/N window {window}.")

    p16, median, p84 = np.nanpercentile(flux[use], [16, 50, 84])
    scatter = 0.5 * (p84 - p16)

    if not np.isfinite(scatter) or scatter <= 0:
        raise ValueError("Could not obtain a positive noise estimate.")

    return median / scatter

# TODO: inspect and adjust these windows.
blue_snr_window = (4600, 4700)
red_snr_window = (5820, 5850)

snr_blue = estimate_snr(blue["wavelength"], blue_flux, blue_snr_window)
snr_red = estimate_snr(red["wavelength"], red_flux, red_snr_window)

print(f"Approximate blue-arm S/N: {snr_blue:.1f}")
print(f"Approximate red-arm S/N:  {snr_red:.1f}")

blue_uncertainty = np.maximum(np.abs(blue_flux) / snr_blue, np.finfo(float).eps)
red_uncertainty = np.maximum(np.abs(red_flux) / snr_red, np.finfo(float).eps)

### Checkpoint 2: extraction choices

Record:

- your final blue and red apertures;
- your adopted S/N windows;
- your estimated S/N values;
- one limitation of this extraction.

**Checkpoint 2 response**

...

# 3. Match and join the blue and red arms

The two arms may overlap in wavelength but differ in flux scale because of imperfect calibration, slit losses, atmospheric effects, or extraction choices.

First plot both arms together. Then use their overlap to estimate a multiplicative scale factor. Do not simply hide a discontinuity without checking whether the spectral shapes are mutually consistent.

In [ ]:
fig, ax = plt.subplots()
ax.plot(blue["wavelength"], blue_flux, label="Blue arm")
ax.plot(red["wavelength"], red_flux, label="Red arm")
ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Extracted flux")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
def estimate_arm_scale(
    blue_wave,
    blue_flux,
    red_wave,
    red_flux,
    overlap_window,
    n_points=500,
):
    lower = max(overlap_window[0], np.nanmin(blue_wave), np.nanmin(red_wave))
    upper = min(overlap_window[1], np.nanmax(blue_wave), np.nanmax(red_wave))

    if upper <= lower:
        raise ValueError("The selected arms do not overlap in this wavelength window.")

    common_wave = np.linspace(lower, upper, n_points)
    blue_common = np.interp(common_wave, blue_wave, blue_flux)
    red_common = np.interp(common_wave, red_wave, red_flux)

    valid = (
        np.isfinite(blue_common)
        & np.isfinite(red_common)
        & (red_common != 0)
    )
    ratio = blue_common[valid] / red_common[valid]

    return np.nanmedian(ratio), common_wave, blue_common, red_common

# TODO: choose an overlap window supported by both arms.
overlap_window = (5500, 5650)

red_scale, overlap_wave, overlap_blue, overlap_red = estimate_arm_scale(
    blue["wavelength"],
    blue_flux,
    red["wavelength"],
    red_flux,
    overlap_window,
)

print(f"Multiply the red arm by approximately {red_scale:.3f}")

fig, ax = plt.subplots()
ax.plot(overlap_wave, overlap_blue, label="Blue")
ax.plot(overlap_wave, red_scale * overlap_red, label="Scaled red")
ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Flux")
ax.set_title("Check the arm match in the overlap")
ax.legend()
fig.tight_layout()
plt.show()

Choose a join wavelength inside the reliable overlap. The simple method below keeps the blue arm below the join and the scaled red arm above it.

This throws away part of the overlap. An optional challenge is to blend both arms using their uncertainties.

In [ ]:
# TODO: choose the useful wavelength limits and join point after inspection.
join_wavelength = 5600.0
blue_use = (blue["wavelength"] >= 3500) & (blue["wavelength"] <= join_wavelength)
red_use = red["wavelength"] > join_wavelength

combined = Table()
combined["wavelength"] = np.concatenate(
    [blue["wavelength"][blue_use], red["wavelength"][red_use]]
)
combined["flux"] = np.concatenate(
    [blue_flux[blue_use], red_scale * red_flux[red_use]]
)
combined["uncertainty"] = np.concatenate(
    [blue_uncertainty[blue_use], red_scale * red_uncertainty[red_use]]
)
combined.sort("wavelength")

spectrum = {
    "object": blue["object"],
    "wavelength": np.asarray(combined["wavelength"]),
    "flux": np.asarray(combined["flux"]),
    "uncertainty": np.asarray(combined["uncertainty"]),
}

fig, ax = plt.subplots()
ax.plot(spectrum["wavelength"], spectrum["flux"])
ax.axvline(join_wavelength, linestyle="--", label="Arm join")
ax.set_xlabel(r"Wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Flux")
ax.set_title(f"Combined spectrum: {spectrum['object']}")
ax.legend()
fig.tight_layout()
plt.show()

### Checkpoint 3: arm matching

- What red-arm scale did you obtain?
- Is a single multiplicative scale adequate across the overlap?
- Can you see the join in the combined spectrum?
- What would be a better combination method?

**Checkpoint 3 response**

...

# 4. Estimate the radial velocity

A star's line-of-sight motion shifts its stellar absorption lines. Positive radial velocity conventionally means the star is receding and its lines are redshifted.

For the velocities in this exercise, the classical approximation is often adequate, but the helper below uses the relativistic Doppler relation.

Telluric absorption is produced in Earth's atmosphere and should **not** be shifted into the stellar rest frame.

In [ ]:
REST_LINES = {
    "Ca K": 3933.66,
    "Ca H": 3968.47,
    r"H$\delta$": 4101.74,
    r"H$\gamma$": 4340.47,
    r"H$\beta$": 4861.33,
    "Mg b": 5172.68,
    "Na D": 5891.58,
    r"H$\alpha$": 6562.80,
}

SPEED_OF_LIGHT_KMS = c.to_value(u.km / u.s)

def observed_to_rest_wavelength(wavelength_observed, radial_velocity_kms):
    beta = radial_velocity_kms / SPEED_OF_LIGHT_KMS
    if np.any(np.abs(beta) >= 1):
        raise ValueError("Radial velocity must be smaller than the speed of light.")
    doppler_factor = np.sqrt((1 + beta) / (1 - beta))
    return wavelength_observed / doppler_factor

def plot_line_windows(spectrum, radial_velocity_kms=0.0, half_width=18):
    wavelength_rest = observed_to_rest_wavelength(
        spectrum["wavelength"], radial_velocity_kms
    )

    selected_lines = ["Ca K", r"H$\gamma$", r"H$\beta$", "Mg b", r"H$\alpha$"]
    fig, axes = plt.subplots(1, len(selected_lines), figsize=(15, 3), sharey=False)

    for ax, line_name in zip(axes, selected_lines):
        rest_wavelength = REST_LINES[line_name]
        use = np.abs(wavelength_rest - rest_wavelength) <= half_width
        ax.plot(wavelength_rest[use], spectrum["flux"][use])
        ax.axvline(rest_wavelength, linestyle="--")
        ax.set_title(line_name)
        ax.set_xlabel(r"Rest wavelength [$\mathrm{\AA}$]")

    axes[0].set_ylabel("Flux")
    fig.suptitle(f"Applied radial velocity: {radial_velocity_kms:.1f} km/s")
    fig.tight_layout()
    plt.show()

In [ ]:
# Start with zero, then change the value until several stellar lines align.
rv_guess = 150.0  # TODO: estimate the radial velocity in km/s
plot_line_windows(spectrum, radial_velocity_kms=rv_guess, half_width=20)

### Checkpoint 4: manual radial velocity

- What radial velocity aligns several lines simultaneously?
- Do all features agree?
- Which features are broad enough that their centres are hard to judge?
- Could any apparent feature be telluric, interstellar, or instrumental?

**Manual radial-velocity estimate:** ... km/s

# 5. Compare the observation with synthetic spectra

We now need to:

1. shift the observed wavelengths to the stellar rest frame;
2. interpolate each model onto those wavelengths;
3. scale the model to the observed flux;
4. define which pixels to compare;
5. calculate a score.

The score below is the median absolute residual divided by the approximate uncertainty. It is robust, but it is **not a formal $\chi^2$ statistic**.

In [ ]:
def interpolate_model(wavelength_observed_rest, model_flux, model_wave):
    return np.interp(
        wavelength_observed_rest,
        model_wave,
        model_flux,
        left=np.nan,
        right=np.nan,
    )

def scale_model_to_observation(observed_flux, model_flux, mask):
    valid = (
        mask
        & np.isfinite(observed_flux)
        & np.isfinite(model_flux)
        & (model_flux != 0)
    )
    if np.count_nonzero(valid) < 20:
        raise ValueError("Too few valid pixels to scale the model.")
    return np.nanmedian(observed_flux[valid] / model_flux[valid])

def robust_normalised_residual_score(observed_flux, model_flux, uncertainty, mask):
    valid = (
        mask
        & np.isfinite(observed_flux)
        & np.isfinite(model_flux)
        & np.isfinite(uncertainty)
        & (uncertainty > 0)
    )
    if np.count_nonzero(valid) < 20:
        return np.inf
    return np.nanmedian(
        np.abs(observed_flux[valid] - model_flux[valid]) / uncertainty[valid]
    )

def make_fitting_mask(wavelength_rest):
    mask = (
        np.isfinite(wavelength_rest)
        & (wavelength_rest >= 3600)
        & (wavelength_rest <= 6800)
    )

    # Example exclusions: telluric regions or problematic edges.
    for lower, upper in [(6270, 6310), (6850, 6950)]:
        mask &= ~((wavelength_rest >= lower) & (wavelength_rest <= upper))

    return mask

## 5.1 Test one model by eye

Choose an exact parameter combination from the grid. Begin with your physical impression of the spectrum rather than a random number.

Think about:

- the broad spectral shape;
- Balmer-line strengths;
- the density and depth of metal lines;
- gravity-sensitive features;
- possible extinction.

In [ ]:
# TODO: choose valid grid values.
trial_parameters = dict(
    teff=6000,
    logg=4.5,
    fe_h=-1.0,
    c_fe=0.0,
    broad=100.0,
    ebv=0.0,
)

wavelength_rest = observed_to_rest_wavelength(
    spectrum["wavelength"], rv_guess
)
fitting_mask = make_fitting_mask(wavelength_rest)

trial_model = get_synthetic_spectrum(grid, **trial_parameters)
trial_model_on_data = interpolate_model(
    wavelength_rest, trial_model, grid["WAVE"]
)
trial_scale = scale_model_to_observation(
    spectrum["flux"], trial_model_on_data, fitting_mask
)
trial_model_on_data *= trial_scale

trial_score = robust_normalised_residual_score(
    spectrum["flux"],
    trial_model_on_data,
    spectrum["uncertainty"],
    fitting_mask,
)

print("Trial parameters:", trial_parameters)
print(f"Robust normalised-residual score: {trial_score:.3f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(wavelength_rest, spectrum["flux"], label="Observation")
ax.plot(wavelength_rest, trial_model_on_data, label="Synthetic model")
ax.set_xlim(3600, 6800)
ax.set_xlabel(r"Rest wavelength [$\mathrm{\AA}$]")
ax.set_ylabel("Flux")
ax.legend()
fig.tight_layout()
plt.show()

## 5.2 Coarse search over the model grid

A full search can be slow. First test a reproducible random subset of grid points and a small extinction grid.

This result is only a **coarse estimate**. It is limited by the grid spacing, imperfect flux calibration, simple uncertainties, unmodelled tellurics, and the lack of interpolation between grid points.

In [ ]:
rng = np.random.default_rng(56789)

available_indices = np.where(grid["BROAD"] == 100.0)[0]
n_random_models = min(250, len(available_indices))
sampled_indices = rng.choice(
    available_indices, size=n_random_models, replace=False
)

ebv_values = [0.0, 0.1, 0.2, 0.3, 0.4]

results = {
    "grid_index": [],
    "teff": [],
    "logg": [],
    "fe_h": [],
    "c_fe": [],
    "ebv": [],
    "score": [],
}

for ebv in ebv_values:
    for index in sampled_indices:
        model_flux = apply_reddening(
            grid["WAVE"], grid["FLUX"][index], ebv
        )
        model_on_data = interpolate_model(
            wavelength_rest, model_flux, grid["WAVE"]
        )

        try:
            model_scale = scale_model_to_observation(
                spectrum["flux"], model_on_data, fitting_mask
            )
        except ValueError:
            continue

        model_on_data *= model_scale
        score = robust_normalised_residual_score(
            spectrum["flux"],
            model_on_data,
            spectrum["uncertainty"],
            fitting_mask,
        )

        results["grid_index"].append(index)
        results["teff"].append(grid["TEFF"][index])
        results["logg"].append(grid["LOGG"][index])
        results["fe_h"].append(grid["FEH"][index])
        results["c_fe"].append(grid["CFE"][index])
        results["ebv"].append(ebv)
        results["score"].append(score)

results = Table(results)
results.sort("score")
results[:10]

In [ ]:
best = results[0]

best_parameters = dict(
    teff=float(best["teff"]),
    logg=float(best["logg"]),
    fe_h=float(best["fe_h"]),
    c_fe=float(best["c_fe"]),
    broad=100.0,
    ebv=float(best["ebv"]),
)

best_model = get_synthetic_spectrum(grid, **best_parameters)
best_model_on_data = interpolate_model(
    wavelength_rest, best_model, grid["WAVE"]
)
best_scale = scale_model_to_observation(
    spectrum["flux"], best_model_on_data, fitting_mask
)
best_model_on_data *= best_scale

print("Best coarse-search parameters:")
for name, value in best_parameters.items():
    print(f"  {name}: {value}")
print(f"  score: {float(best['score']):.3f}")

## 5.3 Refine the radial velocity using the model

Evaluate a grid of radial velocities near your manual estimate. A minimum in the score indicates the best alignment.

Use the minimum directly. For sub-grid precision, fit a parabola to a few points surrounding the minimum rather than fitting a Gaussian to the entire score curve.

In [ ]:
def score_at_radial_velocity(
    radial_velocity_kms,
    spectrum,
    model_flux,
    model_wave,
):
    rest_wave = observed_to_rest_wavelength(
        spectrum["wavelength"], radial_velocity_kms
    )
    mask = make_fitting_mask(rest_wave)
    model_on_data = interpolate_model(rest_wave, model_flux, model_wave)

    try:
        scale = scale_model_to_observation(
            spectrum["flux"], model_on_data, mask
        )
    except ValueError:
        return np.inf

    model_on_data *= scale
    return robust_normalised_residual_score(
        spectrum["flux"],
        model_on_data,
        spectrum["uncertainty"],
        mask,
    )

# TODO: narrow or widen this range based on your manual estimate.
rv_grid = np.arange(rv_guess - 100, rv_guess + 100.1, 1.0)
rv_scores = np.array([
    score_at_radial_velocity(
        rv, spectrum, best_model, grid["WAVE"]
    )
    for rv in rv_grid
])

best_rv_grid = rv_grid[np.nanargmin(rv_scores)]
print(f"Best grid radial velocity: {best_rv_grid:.1f} km/s")

fig, ax = plt.subplots()
ax.plot(rv_grid, rv_scores)
ax.axvline(best_rv_grid, linestyle="--", label=f"{best_rv_grid:.1f} km/s")
ax.set_xlabel(r"Radial velocity [$\mathrm{km\,s^{-1}}$]")
ax.set_ylabel("Robust comparison score")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Optional: parabolic interpolation around the minimum.
minimum_index = int(np.nanargmin(rv_scores))
half_window = 3

left = max(0, minimum_index - half_window)
right = min(len(rv_grid), minimum_index + half_window + 1)

if right - left >= 3:
    coefficients = np.polyfit(
        rv_grid[left:right],
        rv_scores[left:right],
        deg=2,
    )
    if coefficients[0] > 0:
        best_rv_parabola = -coefficients[1] / (2 * coefficients[0])
        print(f"Parabolic estimate: {best_rv_parabola:.2f} km/s")
    else:
        best_rv_parabola = best_rv_grid
        print("Local parabola did not open upward; using the grid minimum.")
else:
    best_rv_parabola = best_rv_grid
    print("Not enough surrounding points; using the grid minimum.")

### Checkpoint 5: fitting and limitations

Record:

- manual radial velocity;
- refined radial velocity;
- best coarse-grid parameters;
- whether several models have similar scores;
- at least three reasons not to over-interpret the result.

**Checkpoint 5 response**

...

# 6. Plot clinic: turn an analysis into a scientific result

A plot is not automatically informative just because it contains all the data.

Run the deliberately weak plot below and identify its problems. Consider:

- What question is the plot answering?
- Are the axes and units clear?
- Can you distinguish observation, model, and residual?
- Is the wavelength range too broad to inspect line agreement?
- Is the residual shown on a meaningful scale?
- Are masked regions or telluric features identified?
- Does the title contain useful information without becoming unreadable?
- Could the figure be understood outside this notebook?

In [ ]:
final_rv = best_rv_parabola
final_rest_wave = observed_to_rest_wavelength(
    spectrum["wavelength"], final_rv
)
final_mask = make_fitting_mask(final_rest_wave)
final_model_on_data = interpolate_model(
    final_rest_wave, best_model, grid["WAVE"]
)
final_scale = scale_model_to_observation(
    spectrum["flux"], final_model_on_data, final_mask
)
final_model_on_data *= final_scale
final_residual = spectrum["flux"] - final_model_on_data

# Deliberately weak plot: discuss what is wrong with it.
plt.plot(final_rest_wave, spectrum["flux"])
plt.plot(final_rest_wave, final_model_on_data)
plt.plot(final_rest_wave, final_residual)
plt.title("fit")
plt.show()

## 6.1 Build a better diagnostic figure

Create a figure that includes:

1. observation and model in a main panel;
2. residuals in a separate aligned panel;
3. clear axis labels and units;
4. a useful wavelength interval or several zoom panels;
5. a legend;
6. the object identifier and fitted parameters;
7. visual indication of excluded pixels or telluric regions where useful.

Optional additions:

- an uncertainty band;
- labels for important spectral lines;
- separate panels for H$\beta$, Mg b, Na D, and H$\alpha$;
- a note that the parameters are coarse grid estimates.

In [ ]:
# TODO: replace this scaffold with your final diagnostic figure.

fig, axes = plt.subplots(
    2,
    1,
    figsize=(11, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

# Main panel:
# axes[0].plot(...)
# axes[0].plot(...)

# Residual panel:
# axes[1].plot(...)
# axes[1].axhline(...)

# Labels, title, legend, wavelength limits, and any masked regions:
# ...

fig.tight_layout()

# Save a copy for the group discussion.
output_filename = FIGURE_DIR / f"{spectrum['object']}_diagnostic.png"
# fig.savefig(output_filename, dpi=200, bbox_inches="tight")

plt.show()

## 6.2 Compare a broad overview with a line-level view

A full-spectrum plot is useful for the continuum shape and obvious calibration problems. It is usually poor for judging whether individual spectral lines fit.

Make at least one line-level plot. Good choices include:

- Ca H and K;
- H$\gamma$ or H$\beta$;
- Mg b;
- Na D;
- H$\alpha$.

State what new information the zoomed view reveals.

In [ ]:
# TODO: create one or more line-level panels.

# 7. Final interpretation

Prepare a two-minute report for the group:

1. What object did you analyse?
2. What radial velocity did you infer?
3. What coarse stellar parameters did you infer?
4. Which parts of the spectrum drove that interpretation?
5. Where did the model fail?
6. Which analysis step most limits your confidence?
7. Show one plot that supports your conclusions.

## Final results

- Object:
- Radial velocity:
- $T_{\mathrm{eff}}$:
- $\log g$:
- [Fe/H]:
- [C/Fe]:
- $E(B-V)$:
- Most convincing evidence:
- Largest limitation:

# Optional challenges

Choose one:

1. **Aperture test:** repeat the extraction with a smaller and larger aperture. How stable are the spectrum, S/N, and fitted parameters?
2. **Arm combination:** blend the overlap using inverse-variance weights rather than discarding one arm.
3. **Masking:** identify telluric or poorly calibrated regions and test their effect on the fitted parameters.
4. **Full grid:** evaluate all grid points at the best radial velocity and compare with the random subset.
5. **Parameter degeneracies:** plot the score against temperature, gravity, metallicity, and extinction. Which parameters are correlated?
6. **Uncertainty:** use the spread of well-fitting models to describe plausible parameter ranges, while clearly stating that this is not a formal posterior.
7. **Continuum correction:** fit a smooth multiplicative correction between the model and observation, then discuss what astrophysical information it may remove.

# What a production analysis would improve

This practical intentionally simplifies many steps. A research-quality analysis would normally improve:

- sky subtraction and background estimation;
- variance propagation from the detector level;
- bad-pixel and cosmic-ray treatment;
- wavelength-dependent spatial extraction;
- flux calibration and atmospheric extinction;
- telluric modelling;
- line-spread-function and spectral-resolution matching;
- synthetic-grid interpolation;
- continuum treatment;
- radial-velocity optimisation;
- parameter uncertainties and covariances;
- external information from photometry, parallax, and stellar-evolution models.

A simple result can still be scientifically useful when its assumptions and limitations are made explicit.